# Imbalanced Data: Resampling, Class Weights, and Focal Loss

Class imbalance is ubiquitous in real-world ML: fraud detection (0.1%), medical diagnosis, spam filtering, defect detection.

## Strategy Overview

| Strategy | Approach | Pros | Cons |
|----------|----------|------|------|
| **Random oversampling** | Duplicate minority samples | Simple | Overfitting to duplicates |
| **Random undersampling** | Remove majority samples | Simple, faster training | Loses information |
| **SMOTE** | Synthesize new minority samples | Generates new examples | Can create noise, doesn't work well in high dimensions |
| **Class weights** | Increase loss for minority class | No data manipulation | May not be enough for extreme imbalance |
| **Focal loss** | Down-weight easy examples | Focuses on hard examples | Extra hyperparameter (gamma) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             precision_recall_curve, f1_score)

np.random.seed(42)

## Create Highly Imbalanced Dataset

In [ ]:
X, y = make_classification(
    n_samples=5000, n_features=10, n_informative=5,
    n_classes=2, weights=[0.95, 0.05],  # 95% negative, 5% positive
    flip_y=0.02, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

print(f"Training set: {len(y_train)} samples")
print(f"  Class 0 (majority): {(y_train == 0).sum()} ({(y_train == 0).mean():.1%})")
print(f"  Class 1 (minority): {(y_train == 1).sum()} ({(y_train == 1).mean():.1%})")
print(f"  Imbalance ratio: {(y_train == 0).sum() / (y_train == 1).sum():.1f}:1")

## 1. Random Oversampling

In [ ]:
def random_oversample(X, y, target_ratio=1.0):
    """Oversample minority class by random duplication."""
    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    minority_class = classes[np.argmin(counts)]
    
    n_majority = counts.max()
    n_target = int(n_majority * target_ratio)
    
    minority_idx = np.where(y == minority_class)[0]
    n_to_add = n_target - len(minority_idx)
    
    if n_to_add <= 0:
        return X, y
    
    # Randomly duplicate minority samples
    oversample_idx = np.random.choice(minority_idx, n_to_add, replace=True)
    X_new = np.vstack([X, X[oversample_idx]])
    y_new = np.concatenate([y, y[oversample_idx]])
    
    return X_new, y_new

X_over, y_over = random_oversample(X_train, y_train)
print(f"After oversampling: {len(y_over)} samples")
print(f"  Class 0: {(y_over == 0).sum()}, Class 1: {(y_over == 1).sum()}")

## 2. Random Undersampling

In [ ]:
def random_undersample(X, y, target_ratio=1.0):
    """Undersample majority class by random removal."""
    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    minority_class = classes[np.argmin(counts)]
    
    n_minority = counts.min()
    n_target = int(n_minority / target_ratio)
    
    majority_idx = np.where(y == majority_class)[0]
    minority_idx = np.where(y == minority_class)[0]
    
    # Randomly select subset of majority
    keep_idx = np.random.choice(majority_idx, n_target, replace=False)
    all_idx = np.concatenate([keep_idx, minority_idx])
    np.random.shuffle(all_idx)
    
    return X[all_idx], y[all_idx]

X_under, y_under = random_undersample(X_train, y_train)
print(f"After undersampling: {len(y_under)} samples")
print(f"  Class 0: {(y_under == 0).sum()}, Class 1: {(y_under == 1).sum()}")
print(f"  Lost {len(y_train) - len(y_under)} majority samples!")

## 3. SMOTE (Synthetic Minority Oversampling Technique)

**Algorithm:**
1. For each minority sample $x_i$:
2. Find its $k$ nearest minority neighbors
3. Randomly pick a neighbor $x_j$
4. Create synthetic sample: $x_{\text{new}} = x_i + \lambda \cdot (x_j - x_i)$ where $\lambda \sim U(0,1)$

This creates new points along the line segment between existing minority samples.

In [ ]:
def smote(X, y, k=5, target_ratio=1.0):
    """SMOTE: Synthetic Minority Oversampling Technique."""
    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    minority_class = classes[np.argmin(counts)]
    
    minority_idx = np.where(y == minority_class)[0]
    X_minority = X[minority_idx]
    
    n_majority = counts.max()
    n_to_generate = int(n_majority * target_ratio) - len(minority_idx)
    
    if n_to_generate <= 0:
        return X, y
    
    # Compute pairwise distances among minority samples
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k + 1)  # +1 because includes self
    nn.fit(X_minority)
    distances, indices = nn.kneighbors(X_minority)
    
    synthetic = []
    for _ in range(n_to_generate):
        # Pick a random minority sample
        i = np.random.randint(len(X_minority))
        # Pick a random neighbor (skip index 0 = self)
        neighbor_idx = indices[i, np.random.randint(1, k + 1)]
        # Interpolate
        lam = np.random.random()
        new_sample = X_minority[i] + lam * (X_minority[neighbor_idx] - X_minority[i])
        synthetic.append(new_sample)
    
    synthetic = np.array(synthetic)
    X_new = np.vstack([X, synthetic])
    y_new = np.concatenate([y, np.full(len(synthetic), minority_class)])
    
    return X_new, y_new

X_smote, y_smote = smote(X_train, y_train, k=5)
print(f"After SMOTE: {len(y_smote)} samples")
print(f"  Class 0: {(y_smote == 0).sum()}, Class 1: {(y_smote == 1).sum()}")

## 4. Class Weights in Loss Function

Instead of modifying data, modify the loss to penalize minority misclassification more:

$$L = -\sum_i \left[ w_{y_i} \cdot y_i \log(\hat{y}_i) + w_{y_i} \cdot (1-y_i)\log(1-\hat{y}_i) \right]$$

Typical weighting: $w_c = \frac{n}{C \cdot n_c}$ (inverse of class frequency).

In [ ]:
def compute_class_weights(y):
    """Compute balanced class weights: n / (n_classes * n_c)."""
    classes, counts = np.unique(y, return_counts=True)
    n = len(y)
    n_classes = len(classes)
    weights = {c: n / (n_classes * count) for c, count in zip(classes, counts)}
    return weights

class_weights = compute_class_weights(y_train)
print("Class weights:")
for cls, w in class_weights.items():
    print(f"  Class {cls}: {w:.3f}")
print(f"  Minority class gets {class_weights[1]/class_weights[0]:.1f}x more weight")

## 5. Focal Loss

From the RetinaNet paper. Adds a modulating factor that down-weights easy examples:

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- $p_t$ = model's estimated probability for the true class
- $\gamma$ = focusing parameter (typically 2). Higher $\gamma$ = more focus on hard examples
- $\alpha_t$ = class weighting factor

When $\gamma = 0$: standard cross-entropy. When $\gamma > 0$: well-classified examples get very low loss.

In [ ]:
def focal_loss(y_true, y_pred_prob, gamma=2.0, alpha=None):
    """Compute focal loss.
    
    y_pred_prob: predicted probability for class 1.
    """
    eps = 1e-7
    y_pred_prob = np.clip(y_pred_prob, eps, 1 - eps)
    
    # p_t: probability of the true class
    p_t = np.where(y_true == 1, y_pred_prob, 1 - y_pred_prob)
    
    # Focal modulating factor
    focal_weight = (1 - p_t) ** gamma
    
    # Alpha weighting
    if alpha is not None:
        alpha_t = np.where(y_true == 1, alpha, 1 - alpha)
    else:
        alpha_t = 1.0
    
    loss = -alpha_t * focal_weight * np.log(p_t)
    return loss.mean()

# Compare CE vs Focal Loss across probability range
p = np.linspace(0.01, 0.99, 100)

fig, ax = plt.subplots(figsize=(10, 5))

# For a positive example (y=1)
ce_loss = -np.log(p)
fl_gamma1 = -(1 - p)**1 * np.log(p)
fl_gamma2 = -(1 - p)**2 * np.log(p)
fl_gamma5 = -(1 - p)**5 * np.log(p)

ax.plot(p, ce_loss, label='CE (gamma=0)', linewidth=2)
ax.plot(p, fl_gamma1, label='Focal (gamma=1)', linewidth=2)
ax.plot(p, fl_gamma2, label='Focal (gamma=2)', linewidth=2)
ax.plot(p, fl_gamma5, label='Focal (gamma=5)', linewidth=2)

ax.set_xlabel('Predicted probability for true class')
ax.set_ylabel('Loss')
ax.set_title('Cross-Entropy vs Focal Loss (for a positive example)')
ax.legend()
ax.set_ylim(0, 5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: Focal loss makes well-classified examples (high p_t)")
print("contribute almost zero loss, letting the model focus on hard cases.")

## Comparison: Decision Boundaries Before/After Resampling

In [ ]:
# Create 2D imbalanced data for visualization
np.random.seed(42)
n_maj = 500
n_min = 25

X_maj = np.random.randn(n_maj, 2) * 1.5
X_min = np.random.randn(n_min, 2) * 0.8 + np.array([2.5, 2.5])

X_2d = np.vstack([X_maj, X_min])
y_2d = np.array([0]*n_maj + [1]*n_min)

# Apply different strategies
X_2d_over, y_2d_over = random_oversample(X_2d, y_2d)
X_2d_under, y_2d_under = random_undersample(X_2d, y_2d)
X_2d_smote, y_2d_smote = smote(X_2d, y_2d, k=5)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

datasets = [
    (X_2d, y_2d, 'Original (imbalanced)'),
    (X_2d_over, y_2d_over, 'Random Oversampling'),
    (X_2d_under, y_2d_under, 'Random Undersampling'),
    (X_2d_smote, y_2d_smote, 'SMOTE'),
]

for ax, (X_d, y_d, title) in zip(axes.flat, datasets):
    # Train model
    model = LogisticRegression()
    model.fit(X_d, y_d)
    
    # Decision boundary
    h = 0.1
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    ax.scatter(X_d[y_d == 0, 0], X_d[y_d == 0, 1], c='blue', s=10, alpha=0.3, label='Majority')
    ax.scatter(X_d[y_d == 1, 0], X_d[y_d == 1, 1], c='red', s=30, alpha=0.6, label='Minority')
    
    # Evaluate on original data
    y_pred = model.predict(X_2d)
    f1 = f1_score(y_2d, y_pred)
    n0 = (y_d == 0).sum()
    n1 = (y_d == 1).sum()
    ax.set_title(f'{title}\n(n0={n0}, n1={n1}, F1={f1:.3f})')
    ax.legend(fontsize=8)

plt.suptitle('Decision Boundaries under Different Resampling Strategies', fontsize=14)
plt.tight_layout()
plt.show()

## Full Comparison on Test Set

In [ ]:
strategies = {
    'Baseline (no resampling)': (X_train, y_train, {}),
    'Random Oversampling': (X_over, y_over, {}),
    'Random Undersampling': (X_under, y_under, {}),
    'SMOTE': (X_smote, y_smote, {}),
    'Class Weights': (X_train, y_train, {'class_weight': 'balanced'}),
}

print(f"{'Strategy':<30} {'F1':<8} {'Precision':<10} {'Recall':<10} {'AUC-ROC'}")
print("-" * 75)

for name, (X_tr, y_tr, kwargs) in strategies.items():
    model = LogisticRegression(max_iter=1000, **kwargs)
    model.fit(X_tr, y_tr)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    f1 = f1_score(y_test, y_pred)
    precision = (y_pred[y_test == 1] == 1).sum() / max(y_pred.sum(), 1)
    recall = (y_pred[y_test == 1] == 1).sum() / (y_test == 1).sum()
    auc = roc_auc_score(y_test, y_prob)
    
    print(f"{name:<30} {f1:<8.4f} {precision:<10.4f} {recall:<10.4f} {auc:.4f}")

## Notes: Threshold Tuning

Instead of using the default threshold of 0.5, optimize the decision threshold
on the precision-recall curve for your business objective.

In [ ]:
# Train with class weights and tune threshold
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Find threshold that maximizes F1
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recalls, precisions, linewidth=2)
axes[0].scatter(recalls[best_idx], precisions[best_idx], color='red', s=100, zorder=5,
               label=f'Best F1 @ threshold={best_threshold:.3f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()

axes[1].plot(thresholds, f1_scores, linewidth=2, color='orange')
axes[1].axvline(x=best_threshold, color='red', linestyle='--',
               label=f'Best threshold={best_threshold:.3f}')
axes[1].axvline(x=0.5, color='gray', linestyle=':', label='Default (0.5)')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score vs Decision Threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Default threshold (0.5): F1 = {f1_score(y_test, (y_prob >= 0.5).astype(int)):.4f}")
print(f"Optimal threshold ({best_threshold:.3f}): F1 = {f1_scores[best_idx]:.4f}")

## Interview Questions

### Q: SMOTE -- how does it work?
**A:** For each minority sample, find its k nearest minority-class neighbors. Generate a synthetic point along the line segment between the sample and a randomly chosen neighbor: $x_{new} = x_i + \lambda(x_j - x_i)$, $\lambda \in [0,1]$. This creates new, plausible minority examples rather than just duplicating.

**Limitations:** Can create noisy samples in overlapping regions. Doesn't work well in very high dimensions (distances become meaningless). Variants: Borderline-SMOTE (only oversample near decision boundary), ADASYN (adaptive).

### Q: Oversampling vs undersampling tradeoffs?
**A:**
- **Oversampling:** Preserves all data, but risk overfitting to duplicated samples. More training data = slower.
- **Undersampling:** Faster training, but throws away potentially useful majority samples. Can be combined with ensemble methods (train multiple classifiers on different undersampled subsets).
- **In practice:** Start with class weights (simplest), then try SMOTE. For extreme imbalance, combine undersampling majority + SMOTE minority.

### Q: When to use class weights vs resampling?
**A:**
- **Class weights** are simpler, don't change data distribution, and work with any loss-based model. Preferred for moderate imbalance.
- **Resampling** is more flexible: SMOTE generates novel points, undersampling reduces compute. Better for extreme imbalance (>100:1).
- **Focal loss** when you suspect the problem is not just imbalance but also easy/hard example distribution.

### Notes on Cost-Sensitive Learning
In many real applications (fraud, medical), false negatives and false positives have different costs. Build a cost matrix:
```
           Predicted 0    Predicted 1
Actual 0   $0 (TN)        $10 (FP: investigate cost)
Actual 1   $1000 (FN:     $0 (TP)
            missed fraud)
```
Then optimize the expected cost, not just accuracy.